In [13]:
import polars as pl
from project_paths import paths
from src.exposure_aggregate import load_anchors, WINDOWS, STATS, window_label


In [14]:

features = pl.read_parquet(paths.processed_data / "exposure_features.parquet")
anchors = load_anchors()


In [15]:
features["asset_id"].sort().equals(anchors["asset_id"].sort())



True

In [16]:

features.shape

(23304, 36)

In [17]:
subsets = ["all", "matched"]
labels = [window_label(w) for w in WINDOWS]          


In [18]:

def col(subset, label, stat):
    return f"exp__{subset}__{label}__{stat}"

In [19]:
stat_cols = {col(subset, label, stat) for subset in subsets for label in labels for stat in STATS}

exp_cols = {column for column in features.columns if column.startswith("exp__")}
flag_cols = {column for column in exp_cols if column.endswith("window_truncated")}




In [20]:
print(stat_cols - exp_cols)


set()


ought to be no columns...

fixed :)

In [21]:
print((exp_cols - stat_cols - flag_cols))


set()


In [22]:


sorted(flag_cols)

['exp__15y__window_truncated',
 'exp__2y__window_truncated',
 'exp__5y__window_truncated']

In [23]:
schema = features.schema

print(all(schema[col(subset, label, "n_events")] == pl.Int64 for subset in subsets for label in labels))



True


In [24]:

print(all(
    schema[col(subset, label, stat)] == pl.Float64
    for subset in subsets for label in labels for stat in ("freq", "cum_area", "max_event_area")
))


True


In [25]:

print(
    all(schema[col] == pl.Boolean for col in flag_cols)
)

True


In [26]:
neg = {c: features.select((pl.col(c) < 0).sum()).item() for c in stat_cols}

print(any(neg.values()))



False


In [27]:

float_cols = [c for c in stat_cols if schema[c] == pl.Float64]
inf = {c: features.select(pl.col(c).is_infinite().sum()).item() for c in float_cols}

print(any(inf.values()))

False


In [28]:
violations = {}
for subset in subsets:
    for stat in ["n_events", "cum_area", "max_event_area"]:
        cols = [col(subset, label, stat) for label in labels]        
        bad = features.select(
            pl.any_horizontal(pl.col(b) < pl.col(a) for a, b in zip(cols, cols[1:]))
            .fill_null(False).sum()
        ).item()
        if bad:
            violations[(subset, stat)] = bad

violations

{('all', 'cum_area'): 9, ('matched', 'cum_area'): 10}

In [29]:
for label in labels:
    for stat in STATS:
        all = col("all", label, stat)
        matched = col("matched", label, stat)
        bad = features.select((pl.col(matched) > pl.col(all)).fill_null(False).sum()).item()
        
        if bad:
            violations[(label, stat)] = bad

violations

{('all', 'cum_area'): 9,
 ('matched', 'cum_area'): 10,
 ('2y', 'cum_area'): 1,
 ('5y', 'cum_area'): 1,
 ('15y', 'cum_area'): 29,
 ('lifetime', 'cum_area'): 168}

the cumulative area should always be:
    1. the "all" version should always be greater than or equal the "matched" option. "matched" should always be a subset of the polygons, and at most the same set of polygons as "all". 
    2. the cumulative area should always be greater or the same as the window increases, never less. 

My instinct is that these cases will be unequal only in very small decimal points caused by some non determinism in float opperations. Rounding to a sensible number of decimal points might resolve the violations.

In [38]:

for label in labels:
    for stat in STATS:
        all = col("all", label, stat)
        matched = col("matched", label, stat)
        bad = features.select(
            delta=pl.col(matched) - pl.col(all),
            rel=(pl.col(matched) - pl.col(all)) / pl.col(all),
        ).filter(pl.col("delta") > 0)
        
        if bad.height > 0:
            print(label, stat)
            print(bad)


2y cum_area
shape: (1, 2)
┌────────────┬────────────┐
│ delta      ┆ rel        │
│ ---        ┆ ---        │
│ f64        ┆ f64        │
╞════════════╪════════════╡
│ 3.6380e-12 ┆ 1.7600e-16 │
└────────────┴────────────┘
5y cum_area
shape: (1, 2)
┌────────────┬────────────┐
│ delta      ┆ rel        │
│ ---        ┆ ---        │
│ f64        ┆ f64        │
╞════════════╪════════════╡
│ 2.2737e-13 ┆ 1.5147e-16 │
└────────────┴────────────┘
15y cum_area
shape: (29, 2)
┌────────────┬────────────┐
│ delta      ┆ rel        │
│ ---        ┆ ---        │
│ f64        ┆ f64        │
╞════════════╪════════════╡
│ 1.4552e-11 ┆ 1.5015e-16 │
│ 9.0949e-13 ┆ 1.1845e-16 │
│ 3.6380e-12 ┆ 2.0861e-16 │
│ 9.0949e-13 ┆ 1.4079e-16 │
│ 3.6380e-12 ┆ 1.6960e-16 │
│ …          ┆ …          │
│ 1.1369e-13 ┆ 1.1357e-16 │
│ 1.1369e-13 ┆ 2.1944e-16 │
│ 5.6843e-14 ┆ 1.4407e-16 │
│ 5.6843e-14 ┆ 2.2140e-16 │
│ 4.5475e-13 ┆ 1.3970e-16 │
└────────────┴────────────┘
lifetime cum_area
shape: (168, 2)
┌────────────┬────

In [40]:
for subset in subsets:
    for stat in ["n_events", "cum_area", "max_event_area"]:
        cols = [col(subset, label, stat) for label in labels]        
        for a, b in zip(cols, cols[1:]):

            bad = features.select(
                delta=pl.col(a) - pl.col(b),
                rel=(pl.col(a) - pl.col(b)) / pl.col(b),
            ).filter(pl.col("delta") > 0)

            if bad.height > 0:
                print(subset, stat, a, b)
                print(bad)


all cum_area exp__all__2y__cum_area exp__all__5y__cum_area
shape: (1, 2)
┌────────────┬────────────┐
│ delta      ┆ rel        │
│ ---        ┆ ---        │
│ f64        ┆ f64        │
╞════════════╪════════════╡
│ 2.2737e-13 ┆ 1.5147e-16 │
└────────────┴────────────┘
all cum_area exp__all__5y__cum_area exp__all__15y__cum_area
shape: (1, 2)
┌────────────┬────────────┐
│ delta      ┆ rel        │
│ ---        ┆ ---        │
│ f64        ┆ f64        │
╞════════════╪════════════╡
│ 9.0949e-13 ┆ 1.4681e-16 │
└────────────┴────────────┘
all cum_area exp__all__15y__cum_area exp__all__lifetime__cum_area
shape: (7, 2)
┌────────────┬────────────┐
│ delta      ┆ rel        │
│ ---        ┆ ---        │
│ f64        ┆ f64        │
╞════════════╪════════════╡
│ 1.1369e-13 ┆ 2.0208e-16 │
│ 5.6843e-14 ┆ 1.8911e-16 │
│ 1.8190e-12 ┆ 1.1460e-16 │
│ 3.6380e-12 ┆ 1.8358e-16 │
│ 1.8190e-12 ┆ 1.6411e-16 │
│ 9.0949e-13 ┆ 1.4797e-16 │
│ 2.2737e-13 ┆ 2.1832e-16 │
└────────────┴────────────┘
matched cum_area 

this confirms that the differences are tiny. all diffs are in mangnitude of e-12 or smaller. This can be cleaned by rounding to 10 decimal points, but the test confirms the field are usable. 
 

In [30]:
chunk = features.join(
    anchors.select("asset_id", "t_anchor", "protection_type_missing"),
    on="asset_id", how="left",
)
null_cond = {
    "all": pl.col("t_anchor").is_null(),
    "matched": pl.col("t_anchor").is_null() | pl.col("protection_type_missing"),
}

for subset in subsets:
    for label in labels:
        for st in STATS:
            column = col(subset, label, st)
            bad = chunk.select((pl.col(column).is_null() != null_cond[subset]).sum()).item()

            if bad > 0:
                print(column, bad)

exp__all__2y__freq 13
exp__all__5y__freq 13
exp__all__15y__freq 13
exp__all__lifetime__freq 13
exp__matched__2y__freq 13
exp__matched__5y__freq 13
exp__matched__15y__freq 13
exp__matched__lifetime__freq 13


In [31]:

chunk.select(
    n_assets=pl.len(),
    assessable_all=(~null_cond["all"]).sum(),
    assessable_matched=(~null_cond["matched"]).sum(),
)

n_assets,assessable_all,assessable_matched
u32,u32,u32
23304,23303,23019


In [32]:
features.select(
    pl.col(column).mean().alias(column) for column in sorted(flag_cols)
)

exp__15y__window_truncated,exp__2y__window_truncated,exp__5y__window_truncated
f64,f64,f64
0.098957,0.006952,0.021199


In [ ]:
bad_ids = chunk.filter(
    pl.col(col("all", "lifetime", "freq")).is_null() & ~null_cond["all"]
)["asset_id"]

null_frequency_rows = anchors.filter(pl.col("asset_id").is_in(bad_ids.implode())).select(
    "asset_id", "t_anchor", "asset_start_date"
).join(
    features.select("asset_id", col("all", "lifetime", "n_events")),
    on="asset_id",
)

with pl.Config(tbl_rows=-1):
    print(null_frequency_rows)

shape: (13, 4)
┌──────────┬────────────┬──────────────────┬──────────────────────────────┐
│ asset_id ┆ t_anchor   ┆ asset_start_date ┆ exp__all__lifetime__n_events │
│ ---      ┆ ---        ┆ ---              ┆ ---                          │
│ i64      ┆ date       ┆ date             ┆ i64                          │
╞══════════╪════════════╪══════════════════╪══════════════════════════════╡
│ 178814   ┆ 2018-04-16 ┆ 2024-06-01       ┆ 1                            │
│ 2372     ┆ 2019-06-28 ┆ 2025-06-01       ┆ 1                            │
│ 2494     ┆ 2019-06-28 ┆ 2024-06-01       ┆ 1                            │
│ 25168    ┆ 2024-05-21 ┆ 2024-06-01       ┆ 0                            │
│ 25170    ┆ 2024-02-18 ┆ 2024-06-01       ┆ 0                            │
│ 25171    ┆ 2024-02-18 ┆ 2024-06-01       ┆ 0                            │
│ 39397    ┆ 2024-05-21 ┆ 2024-06-01       ┆ 1                            │
│ 39398    ┆ 2024-02-18 ┆ 2024-06-01       ┆ 0                           

C:\Users\Daniel\AppData\Local\Temp\ipykernel_58800\2269076933.py:5: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  null_frequency_rows = anchors.filter(pl.col("asset_id").is_in(bad_ids)).select(


the obviously wrong one here is asset start date = 2070-04-01. It must be a misformed date. I need to check whether this is also present in the prepared features and this row should be dropped if so and not an easy fix.  



overall, these tests validate the extraction approach, and the coverage is nearly perfect.
All the columns are useable, including cumulative area which just needs rounding to pass the greater than or equal to test but is useable as is.
Apart from the one clearly corrupt value described above, there are 12 other null frequencies which seem to come from uncertainty in the asset start date feature (known already).


Im going to write all of the tests above as asserts - if the notebook is rerun, this cell should pass. If data have changed, this cell will fail if there is an error and stop the notebook finishing.

In [43]:
del all


In [45]:


assert not (stat_cols - exp_cols) and not (exp_cols - stat_cols - flag_cols)

assert all(schema[col(subset, label, "n_events")] == pl.Int64 for subset in subsets for label in labels)

assert all(schema[col(subset, label, st)] == pl.Float64 for subset in subsets for label in labels for st in ("freq", "cum_area", "max_event_area"))

assert all(schema[column] == pl.Boolean for column in flag_cols)

assert not any(features.select((pl.col(column) < 0).sum()).item() for column in stat_cols)

assert not any(features.select(pl.col(column).is_infinite().sum()).item() for column in float_cols)

# use tolerance of 1e-9 to avoid float differences in small decimal point tripping the comparison tests

for subset in subsets:
    for stat in ["n_events", "cum_area", "max_event_area"]:
        cs = [col(subset, label, stat) for label in labels]
        for a, b in zip(cs, cs[1:]):
            bad = features.select(
                ((pl.col(a) - pl.col(b)) > pl.col(b).abs() * 1e-9).fill_null(False).sum()
            ).item()
            assert bad == 0

for label in labels:
    for stat in STATS:
        a, m = col("all", label, stat), col("matched", label, stat)
        bad = features.select(
            ((pl.col(m) - pl.col(a)) > pl.col(a).abs() * 1e-9).fill_null(False).sum()
        ).item()
        assert bad == 0


deviations = anchors.filter(pl.col("asset_start_date") >= pl.col("t_anchor"))["asset_id"]
for subset in subsets:
    for label in labels:
        for stat in STATS:
            column = col(subset, label, stat)
            off = chunk.filter(pl.col(column).is_null() != null_cond[subset])["asset_id"]
            if stat == "freq":
                assert set(off) <= set(deviations)
            else:
                assert off.len() == 0

print("pass")

pass
